# Make plots

I have copied the MSEED files into Dropbox

In [ ]:
from pathlib import Path
from obspy import read, UTCDateTime, Stream
 
launchtime = UTCDateTime('2026-04-01T22:35:12')
duration = 900
pretrigger = 1800
posttrigger = pretrigger
starttime=launchtime - pretrigger
endtime=launchtime + duration + posttrigger
starttrim = launchtime - 30
endtrim = launchtime + 210

MSEED_PATH = Path('~').expanduser() / 'Dropbox' / 'KSC_paper' / 'Artemis2' 


In [ ]:

st = read(str(MSEED_PATH /  '2026*'))
print(st)

In [ ]:
import numpy as np

def clip_trace(trace, percentile=99.9):
    '''
    Clip the data in this Trace to +/- 2x the given percentile of the absolute value of the data.
    '''
    x = trace.data
    clip_level = np.nanpercentile(np.abs(x), percentile) * 2
    print(f"Clipping {trace.id} at {clip_level:.2f}")
    trace.data = np.clip(trace.data, -clip_level, clip_level)

def clip_stream(stream, percentile=99.9):
    for tr in stream:
        clip_trace(tr, percentile=percentile)

st = st.copy()
clip_stream(st, percentile=99.99)

In [ ]:
import numpy as np

def is_flat_trace(tr, threshold=3.0):
    data = tr.data.astype(float)

    if len(data) == 0:
        return True

    rms = np.sqrt(np.mean(data**2))
    peak = np.max(np.abs(data))

    if rms == 0:
        return True

    ratio = peak / rms

    return ratio < threshold

st_event = Stream()  # copy if needed
for tr in st:
    if is_flat_trace(tr, threshold=3.0):
        print(f"Trace {tr.id} is flat (peak/RMS < 3.0), skipping.")
    else:
        st_event.append(tr)

print(st_event)


In [ ]:
stf = st_event.copy() 
stf.detrend('linear')
stf.filter('highpass', freq=0.1, corners=2, zerophase=True)

In [ ]:
# get a list of stations, and then subset for each one, and plot
stations = sorted(set(tr.stats.station for tr in stf))
print(stations)

In [ ]:
for this_station in stations:
    this_st = stf.select(station=this_station)
    print(this_st)
    this_st.plot(type='relative', starttime=starttrim, endtime=endtrim, show=True, equal_scale=False);

In [ ]:
st_seismic = stf.select(channel='*[HL][ZNE012]') 
print(st_seismic)
st_seismic.plot(type='relative', starttime=starttrim, endtime=endtrim, show=True, equal_scale=False);

st_infrasound = stf.select(channel='*D[F0OI12]') # or '*D[F0I]'
print(st_infrasound)
st_infrasound.plot(type='relative', starttime=starttrim, endtime=endtrim, show=True, equal_scale=False);

Correct for instrument response

For USF stations, we can use flovopy.

For Guralp stations, do we have something or need to use NRL?

For B23, what do we have?

For Gem we need gemlog.

In [ ]:
from flovopy.stationmetadata.build import NRL2inventory

st_trillium_corrected = st_seismic.copy().select(channel="DH*")

for tr in st_trillium_corrected:
    print(tr.id, tr.stats.starttime, tr.stats.endtime)

    inv = NRL2inventory(
        nrl_path=None,   # use a local NRL copy
        net=tr.stats.network,
        sta=tr.stats.station,
        loc=tr.stats.location,
        chans=[tr.stats.channel],        # IMPORTANT: list, not string
        datalogger="Centaur",
        sensor="TCP",
        Vpp=40,
        fsamp=tr.stats.sampling_rate,
        lat=0.0,
        lon=0.0,
        elev=0.0,
        depth=0.0,
        sitename="",
        ondate=tr.stats.starttime,
        offdate=tr.stats.endtime,
        sensitivity=None,
        units=None,
    )

    fs = tr.stats.sampling_rate
    nyq = 0.5 * fs
    pre_filt = (0.01, 0.02, 0.94 * nyq, 0.98 * nyq)

    tr.remove_response(
        inventory=inv,
        output="VEL",
        pre_filt=pre_filt,
        zero_mean=True,
        taper=True,
    )

st_trillium_corrected.plot(
    type="relative",
    starttime=starttrim,
    endtime=endtrim,
    show=True,
    equal_scale=False,
);


In [ ]:
# Now correct the infraBSU sensors
from flovopy.stationmetadata.build import NRL2inventory

st_infrabsu_corrected = st_infrasound.copy().select(channel="DD[012F]")  # or "HDF", "BDF", etc., depending on your naming

for tr in st_infrabsu_corrected:
    print(tr.id, tr.stats.starttime, tr.stats.endtime)

    if tr.stats.station == 'B20':
        print(f"Station {tr.stats.station} has channel {tr.stats.channel}, applying special handling for B20 DD0.")
        if tr.stats.channel == 'DDF':
            # chaparral sensor
            inv = NRL2inventory(
                nrl_path=None,   # strongly preferred over remote NRL
                net=tr.stats.network,
                sta=tr.stats.station,
                loc=tr.stats.location,
                chans=[tr.stats.channel],        # IMPORTANT: must be a list
                datalogger="Centaur",
                sensor="Chaparral",
                Vpp=40,
                fsamp=tr.stats.sampling_rate,
                lat=0.0,
                lon=0.0,
                elev=0.0,
                depth=0.0,
                sitename="",
                ondate=tr.stats.starttime,
                offdate=tr.stats.endtime,
                sensitivity=None,
                units="Pa",                      # only used by your fallback path
            )
        elif tr.stats.channel == 'DD0':
            # special case for B20 DD0, which is 40 Vpp since on same Sensor B as the Chaparral DDF
            inv = NRL2inventory(
                nrl_path=None,   # strongly preferred over remote NRL
                net=tr.stats.network,
                sta=tr.stats.station,
                loc=tr.stats.location,
                chans=[tr.stats.channel],        # IMPORTANT: must be a list
                datalogger="Centaur",
                sensor="infrabsu",
                Vpp=40,
                fsamp=tr.stats.sampling_rate,
                lat=0.0,
                lon=0.0,
                elev=0.0,
                depth=0.0,
                sitename="",
                ondate=tr.stats.starttime,
                offdate=tr.stats.endtime,
                sensitivity=None,
                units="Pa",                      # only used by your fallback path
            )
    else:
        inv = NRL2inventory(
            nrl_path=None,   # strongly preferred over remote NRL
            net=tr.stats.network,
            sta=tr.stats.station,
            loc=tr.stats.location,
            chans=[tr.stats.channel],        # IMPORTANT: must be a list
            datalogger="Centaur",
            sensor="infrabsu",
            Vpp=1,
            fsamp=tr.stats.sampling_rate,
            lat=0.0,
            lon=0.0,
            elev=0.0,
            depth=0.0,
            sitename="",
            ondate=tr.stats.starttime,
            offdate=tr.stats.endtime,
            sensitivity=None,
            units="Pa",                      # only used by your fallback path
        )

    fs = tr.stats.sampling_rate
    nyq = 0.5 * fs

    # Example pre-filter; adjust to your actual infrasound band of interest
    pre_filt = (0.001, 0.005, 0.94 * nyq, 0.98 * nyq)

    tr.remove_response(
        inventory=inv,
        output="DEF",                    # safest for pressure/infrasound
        pre_filt=pre_filt,
        zero_mean=True,
        taper=True,
    )


st_infrabsu_corrected.plot(type="relative", show=True, equal_scale=False, starttime=starttrim, endtime=endtrim);

In [ ]:

import gemlog
st_gem = st_infrasound.copy().select(channel='*D[FOI]')
st_gem_corrected = gemlog.deconvolve_gem_response(st_gem, gain='high') 
st_gem_corrected.plot(type='relative', starttime=starttrim, endtime=endtrim, show=True, equal_scale=False);

In [ ]:
master_stream = Stream()
for st in [st_trillium_corrected, st_infrabsu_corrected, st_gem_corrected]:
    master_stream += st
print(master_stream)
master_stream.write(MSEED_PATH / 'master_stream_corrected.mseed', format='MSEED')
master_stream.plot(type='relative', starttime=starttrim, endtime=endtrim, show=True, equal_scale=False);

In [ ]:
from obspy import read, Stream
if not 'master_stream' in locals():
    master_stream = read(str(MSEED_PATH / 'master_stream_corrected.mseed'))

In [ ]:
seismic_subset = Stream(traces=[master_stream.select(station='B01', channel='DHZ')[0],   master_stream.select(station='B29', channel='DHZ')[0]]).trim(starttime=starttrim, endtime=endtrim)
infrasound_subset = Stream(traces=[master_stream.select(station='B01', channel='DD0')[0],   master_stream.select(station='B29', channel='DD0')[0]]).trim(starttime=starttrim, endtime=endtrim)
seismic_subset.plot(equal_scale=True, outfile=MSEED_PATH / 'near_far_seismic_subset.pdf');
infrasound_subset.plot(equal_scale=True, outfile=MSEED_PATH / 'near_far_infrasound_subset.pdf');


In [ ]:

from flovopy.processing.spectrograms import icewebSpectrogram
iwobj = icewebSpectrogram(seismic_subset)
iwobj.precompute()
iwobj.plot(dbscale=True, equal_scale=True, fmax=250.0, clim=[1e-7, 1e-4], cmap='magma', add_colorbar=False, outfile=MSEED_PATH / 'near_far_seismic_spectrogram.png');

iwobji = icewebSpectrogram(infrasound_subset)
iwobji.precompute()
iwobji.plot(dbscale=True, equal_scale=True, fmax=250.0, clim=[2e-2, 2e1], cmap='magma', add_colorbar=False, outfile=MSEED_PATH / 'near_far_infrasound_spectrogram.png');


In [ ]:
import numpy as np
import pandas as pd

master_stream_infrasound = master_stream.copy().select(channel='*D[FOI012]')
master_stream_seismic = master_stream.copy().select(channel='*[HL][ZNE012]')

for index, this_stream in enumerate([master_stream_infrasound, master_stream_seismic]):
    if index == 0:
        trace_type = 'Infrasound'
        digits = 1
    else:
        trace_type = 'Seismic'
        digits = 6
    print(f"Processing {trace_type} traces:")
    peak_amplitudes = {}
    for tr in this_stream:
        tr.detrend(type='demean')
        data = np.abs(tr.data)
        peak_amplitudes[tr.id] = data.max()
    peak_amplitudes = {k: v for k, v in sorted(peak_amplitudes.items(), key=lambda item: item[1], reverse=True)}
    print('Peak amplitudes for each trace (sorted):')

    df = pd.DataFrame(list(peak_amplitudes.items()), columns=['Trace ID', 'Peak Amplitude'])
    df.to_csv(MSEED_PATH.parent / f'peak_{trace_type.lower()}_amplitudes.csv', index=False)
    df['Peak Amplitude'] = df['Peak Amplitude'].round(digits)
    display(df)
    

In [ ]:
import numpy as np
from obspy import read
from scipy.io.wavfile import write as wavwrite
from scipy.signal import resample

def trace_to_audio(
    tr,
    speedup=100,
    output_wav="output.wav",
    audio_rate=44100,
    normalize=True,
    detrend=True,
    taper=True,
    bandpass=None,
):
    """
    Convert an ObsPy Trace to a WAV audio file by speeding it up.

    Parameters
    ----------
    tr : obspy.Trace
        Input seismic trace.
    speedup : float
        Factor by which to speed up playback.
    output_wav : str
        Output WAV filename.
    audio_rate : int
        Desired WAV sample rate in Hz.
    normalize : bool
        If True, scale data to int16 range.
    detrend : bool
        If True, remove mean and linear trend.
    taper : bool
        If True, apply a short taper.
    bandpass : tuple or None
        Optional (fmin, fmax) in Hz for pre-filtering the seismic data.
    """
    tr = tr.copy()

    if detrend:
        tr.detrend("linear")

    if taper:
        tr.taper(max_percentage=0.02)

    if bandpass is not None:
        fmin, fmax = bandpass
        tr.filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True)

    data = tr.data.astype(np.float64)

    # Remove NaNs/infs if present
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    # Effective sampling rate after speeding up
    effective_rate = tr.stats.sampling_rate * speedup

    # Resample to desired audio rate if needed
    if abs(effective_rate - audio_rate) / audio_rate > 0.01:
        n_out = int(round(len(data) * audio_rate / effective_rate))
        if n_out <= 0:
            raise ValueError("Output sample count is invalid.")
        data = resample(data, n_out)

    if normalize:
        peak = np.max(np.abs(data))
        if peak > 0:
            data = data / peak
        data_int16 = np.int16(data * 32767)
    else:
        data_int16 = np.int16(np.clip(data, -32768, 32767))

    wavwrite(output_wav, audio_rate, data_int16)
    return output_wav
for tr in master_stream:
    output_wav = MSEED_PATH.parent / f'{tr.id}_audio.wav'
    trace_to_audio(tr, speedup=60, output_wav=str(output_wav), audio_rate=44100, normalize=True, detrend=True, taper=True)#, bandpass=(0.1, 240))
    print(f'Created audio file: {output_wav}')